In [4]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [5]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!nvidia-smi

%cd /content
!rm -rf CIRI-FS
!git clone --branch asal/CiriEXT4 https://github.com/isusbu/CIRI-FS.git
%cd /content/CIRI-FS

!git branch --show-current

Mon Aug  3 19:38:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
%cd /content/CIRI-FS
!git branch --show-current

/content/CIRI-FS
asal/CiriEXT4


In [8]:
!pip install -q --upgrade \
    transformers \
    accelerate \
    bitsandbytes \
    sentencepiece \
    protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 30.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.


In [9]:
!pip install -q --force-reinstall "protobuf>=5.29.1,<6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 14.2 MB/s eta 0:00:00


In [10]:
import transformers
import accelerate
import google.protobuf
import torch

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Protobuf:", google.protobuf.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Transformers: 5.14.1
Accelerate: 1.14.0
Protobuf: 5.29.6
PyTorch: 2.11.0+cu128
CUDA available: True


In [12]:
!grep -n -A60 -B5 "class DeepseekGen" ciri/query/llm_gen.py

131-            answerList.append(self.tokenizer.decode(output[input_len:-1], skip_special_tokens=True))
132-        return answerList
133-
134-
135-# DeepseekGen must inherit from BaseGen (was missing entirely)
136:class DeepseekGen(BaseGen):
137-    def __init__(self, args: Dict, config_file: str, model, tokenizer):
138-        # Now properly calls BaseGen.__init__
139-        super().__init__(args, config_file)
140-        # Renamed to avoid overwriting self.model from BaseGen
141-        self.llm_model = model
142-        self.tokenizer = tokenizer
143-        # Auto-detect device instead of hardcoding cuda
144-        self.device = get_device()
145-
146-    def _generate(self) -> List:
147-        message = f"{self.config_file}\n{self.prompt}"
148-        messages = [{'role': 'user', 'content': message}]
149-
150-        # Use tokenizer() after apply_chat_template to get correct tensor format
151-        # apply_chat_template with tokenize=False returns a string first
152-        

In [13]:
!grep -n -A40 -B5 'checkpoint.startswith("deepseek")' ciri/ciri_eng.py

107-        dtype = torch.bfloat16 if device == "cuda" else torch.float32
108-        
109-        logger.info(f"Using device: {device.upper()}")
110-        logger.info(f"Using dtype: {dtype}")
111-
112:        if checkpoint.startswith("deepseek"):
113-            
114-            # Use device-aware device_map
115-            full_checkpoint = f"deepseek-ai/{checkpoint}"
116-            logger.info(f"Loading DeepSeek model: {full_checkpoint}")
117-            # updated using  BitsAndBytesConfig for making run in google colab
118-            from transformers import BitsAndBytesConfig
119-            quant_config = BitsAndBytesConfig(
120-                load_in_4bit=True,
121-                bnb_4bit_quant_type="nf4",
122-                bnb_4bit_compute_dtype=torch.bfloat16,
123-                bnb_4bit_use_double_quant=True,
124-            )
125-            model = AutoModelForCausalLM.from_pretrained(
126-                full_checkpoint,
127-                quantization_config=qua

In [14]:
!grep -n -A10 -B5 'args.model.startswith("deepseek")' ciri/ciri_runner.py

18-        return GPTGen(args, file_content)
19-    elif args.model.startswith("claude"):
20-        return ClaudeGen(args, file_content)
21-    elif args.model.startswith("CodeLLaMa"):
22-        return LlamaGen(args, file_content, model, tokenizer)
23:    elif args.model.startswith("deepseek"):
24-        return DeepseekGen(args, file_content, model, tokenizer)
25-    elif args.model.startswith("Qwen"):
26-        return QwenGen(args, file_content, model, tokenizer)
27-    else:
28-        raise ValueError(f"Model {args.model} is not supported")
29-
30-
31-def _run_analysis(llm_gen) -> tuple[str, Optional[dict[str, str]]]:
32-    while True:
33-        answer_parser = llm_gen.generate()


In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM

checkpoint = "deepseek-ai/deepseek-coder-6.7b-instruct"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    trust_remote_code=True
)

print("Loading model...")

from transformers import BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

print("Loaded successfully!")

Loading tokenizer...


config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.87k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

Loading model...


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Loaded successfully!


In [16]:
import torch

messages = [
    {
        "role": "user",
        "content": "What is EXT4? Answer in one sentence."
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt"
).to(model.device)

input_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[0][input_len:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print(answer)

EXT4 is a Linux file system that is used in many Linux distributions, including Ubuntu. It is a successor to the older EXT3 file system and provides many improvements over EXT3, including better performance, more flexibility, and support for larger file systems.


In [17]:
!sed -n '136,177p' ciri/query/llm_gen.py

class DeepseekGen(BaseGen):
    def __init__(self, args: Dict, config_file: str, model, tokenizer):
        # Now properly calls BaseGen.__init__
        super().__init__(args, config_file)
        # Renamed to avoid overwriting self.model from BaseGen
        self.llm_model = model
        self.tokenizer = tokenizer
        # Auto-detect device instead of hardcoding cuda
        self.device = get_device()

    def _generate(self) -> List:
        message = f"{self.config_file}\n{self.prompt}"
        messages = [{'role': 'user', 'content': message}]

        # Use tokenizer() after apply_chat_template to get correct tensor format
        # apply_chat_template with tokenize=False returns a string first
        # then tokenize it properly to avoid KeyError: 'shape'
        formatted = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = self.tokenizer(
            formatted,
            re

In [18]:
!grep -n "^import time" ciri/query/llm_gen.py

3:import time


In [19]:
from pathlib import Path
import re

path = Path("ciri/query/llm_gen.py")
text = path.read_text()

new_class = '''
class DeepseekGen(BaseGen):
    def __init__(self, args: Dict, config_file: str, model, tokenizer):
        super().__init__(args, config_file)

        self.llm_model = model
        self.tokenizer = tokenizer
        self.device = get_device()

        # Cost measurements
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_generation_time = 0.0
        self.generation_calls = 0

    def _generate(self) -> List:
        message = f"{self.config_file}\\n{self.prompt}"
        messages = [{'role': 'user', 'content': message}]

        formatted = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(
            formatted,
            return_tensors="pt"
        ).to(self.device)

        input_len = inputs["input_ids"].shape[1]
        input_token_count = input_len

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        outputs = self.llm_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.2,
            eos_token_id=10252
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        generation_time = time.perf_counter() - start_time

        answerList = []
        output_token_count = 0

        for output in outputs:
            generated_tokens = output[input_len:-1]

            output_token_count += generated_tokens.numel()

            answerList.append(
                self.tokenizer.decode(
                    generated_tokens,
                    skip_special_tokens=True
                )
            )

        self.generation_calls += 1
        self.total_input_tokens += input_token_count
        self.total_output_tokens += output_token_count
        self.total_generation_time += generation_time

        logger.info(
            "[DeepSeek Cost] "
            f"call={self.generation_calls}, "
            f"input_tokens={input_token_count}, "
            f"output_tokens={output_token_count}, "
            f"total_tokens={input_token_count + output_token_count}, "
            f"generation_time_seconds={generation_time:.4f}"
        )

        logger.info(
            "[DeepSeek Cost Cumulative] "
            f"calls={self.generation_calls}, "
            f"input_tokens={self.total_input_tokens}, "
            f"output_tokens={self.total_output_tokens}, "
            f"total_tokens={self.total_input_tokens + self.total_output_tokens}, "
            f"generation_time_seconds={self.total_generation_time:.4f}"
        )

        return answerList
'''

pattern = r'class DeepseekGen\(BaseGen\):.*?(?=\nclass QwenGen)'
updated, n = re.subn(
    pattern,
    new_class.strip() + "\n\n",
    text,
    flags=re.DOTALL
)

if n != 1:
    raise RuntimeError(f"Expected to replace 1 DeepseekGen class, replaced {n}")

path.write_text(updated)
print("✅ DeepseekGen updated with cost logging.")

✅ DeepseekGen updated with cost logging.


In [20]:
from pathlib import Path

path = Path("ciri/query/llm_gen.py")
text = path.read_text()

# Repair the broken two-line f-string created by the patch
text = text.replace(
    'message = f"{self.config_file}\n{self.prompt}"',
    'message = f"{self.config_file}\\n{self.prompt}"'
)

path.write_text(text)

# Verify syntax
import py_compile
py_compile.compile(str(path), doraise=True)

print("✅ Fixed the f-string and syntax check passed.")

✅ Fixed the f-string and syntax check passed.


In [21]:
!sed -n '145,160p' ciri/query/llm_gen.py

        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_generation_time = 0.0
        self.generation_calls = 0

    def _generate(self) -> List:
        message = f"{self.config_file}\n{self.prompt}"
        messages = [{'role': 'user', 'content': message}]

        formatted = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(


In [22]:
!python -m py_compile ciri/query/llm_gen.py

In [23]:
!grep -n "DeepSeek Cost" ciri/query/llm_gen.py

207:            "[DeepSeek Cost] "
216:            "[DeepSeek Cost Cumulative] "


In [24]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.6 MB/s eta 0:00:00


In [ ]:
!pip install anthropic

In [ ]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_SD/erroneous \
    --output_path \
        icse25_data/results/synthesize_config/ext4_SD/deepseek-coder-6.7b-instruct/zero_shot/erroneous \
    --model deepseek-coder-6.7b-instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 0 \
    --misconfig_shot_num 0 \
    --file_format xml \
    --verbose

2026-07-27 21:07:46 - Ciri - INFO - Using device: CUDA
2026-07-27 21:07:46 - Ciri - INFO - Using dtype: torch.bfloat16
2026-07-27 21:07:46 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
Loading weights: 100% 291/291 [00:54<00:00,  5.34it/s]
2026-07-27 21:08:44 - Ciri - INFO - Model loaded successfully on CUDA!
2026-07-27 21:08:44 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-07-27 21:08:50 - Ciri - INFO - [DeepSeek Cost] call=1, input_tokens=412, output_tokens=26, total_tokens=438, generation_time_seconds=5.5389
2026-07-27 21:08:50 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=1, input_tokens=412, output_tokens=26, total_tokens=438, generation_time_seconds=5.5389
2026-07-27 21:08:55 - Ciri - INFO - [DeepSeek Cost] call=2, input_tokens=412, output_tokens=26, total_tokens=438, generation_time_seconds=4.8383
2026-07-27 21:08:55 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=2, input_tokens=824, output_tokens=52, total_tokens=876, generation_time_

In [ ]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_SD/correct \
    --output_path \
        icse25_data/results/synthesize_config/ext4_SD/deepseek-coder-6.7b-instruct/zero_shot/correct \
    --model deepseek-coder-6.7b-instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 0 \
    --misconfig_shot_num 0 \
    --file_format xml \
    --verbose

2026-07-27 21:13:16 - Ciri - INFO - Using device: CUDA
2026-07-27 21:13:16 - Ciri - INFO - Using dtype: torch.bfloat16
2026-07-27 21:13:16 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
Loading weights: 100% 291/291 [00:54<00:00,  5.33it/s]
2026-07-27 21:14:15 - Ciri - INFO - Model loaded successfully on CUDA!
2026-07-27 21:14:15 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-07-27 21:14:21 - Ciri - INFO - [DeepSeek Cost] call=1, input_tokens=408, output_tokens=26, total_tokens=434, generation_time_seconds=5.6058
2026-07-27 21:14:21 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=1, input_tokens=408, output_tokens=26, total_tokens=434, generation_time_seconds=5.6058
2026-07-27 21:14:26 - Ciri - INFO - [DeepSeek Cost] call=2, input_tokens=408, output_tokens=26, total_tokens=434, generation_time_seconds=4.9619
2026-07-27 21:14:26 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=2, input_tokens=816, output_tokens=52, total_tokens=868, generation_time_

In [ ]:
!python icse25_data/script/result_parser.py \
    --project ext4_SD \
    --model deepseek-coder-6.7b-instruct \
    --mode zero_shot

[Ciri Result] on ext4_SD with deepseek-coder-6.7b-instruct and zero_shot mode
File-Level: Precision: N.A., Recall: 0.00, Accuracy: 0.50, F1: N.A.
Param-Level: Precision: N.A., Recall: 0.00, Accuracy: 0.94, F1: N.A.


In [ ]:
!ls "/content/drive/MyDrive/CIRI_EXT4/Dataset/EXT4_XML_Dataset"

ext4_CCD  ext4_CPD  ext4_SD  Groundtruth


In [ ]:
BENCHMARK = "ext4_CPD"
MODEL = "deepseek-coder-6.7b-instruct"

DATA_ROOT = "/content/drive/MyDrive/CIRI_EXT4/Dataset/EXT4_XML_Dataset"
OUTPUT_ROOT = f"icse25_data/results/synthesize_config/{BENCHMARK}/{MODEL}/zero_shot"

CPD

In [ ]:
BENCHMARK = "ext4_CPD"
MODEL = "deepseek-coder-6.7b-instruct"
MODE = "zero_shot"

print(BENCHMARK, MODEL, MODE)

ext4_CPD deepseek-coder-6.7b-instruct zero_shot


In [ ]:
import os

DRIVE_DATA = os.path.realpath(
    "/content/drive/MyDrive/CIRI_EXT4/Dataset"
)

print(DRIVE_DATA)
print("Exists:", os.path.exists(DRIVE_DATA))

/content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset
Exists: True


In [ ]:
DRIVE_DATA = "/content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset/EXT4_XML_Dataset"

In [ ]:
!cp -r "$DRIVE_DATA/ext4_CPD" \
    icse25_data/datasets/synthesize_config/

!cp "$DRIVE_DATA/Groundtruth/ext4_CPD.tsv" \
    icse25_data/datasets/synthesize_config/ground_truth/

In [ ]:
!ls icse25_data/datasets/synthesize_config/ground_truth

alluxio.tsv  ext4_CCD.tsv  hbase.tsv	postgresql.tsv	zookeeper.tsv
django.tsv   ext4_CPD.tsv  hcommon.tsv	redis.tsv
etcd.tsv     ext4_SD.tsv   hdfs.tsv	yarn.tsv


In [ ]:
!python -m ciri.ciri_eng \
  --input_path "$DATA_ROOT/$BENCHMARK/erroneous" \
  --output_path "$OUTPUT_ROOT/erroneous" \
  --model "$MODEL" \
  --system ext4 \
  --version 1.47.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml \
  --verbose

2026-07-27 22:34:34 - Ciri - INFO - Using device: CUDA
2026-07-27 22:34:34 - Ciri - INFO - Using dtype: torch.bfloat16
2026-07-27 22:34:34 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
Loading weights: 100% 291/291 [00:54<00:00,  5.34it/s]
2026-07-27 22:35:33 - Ciri - INFO - Model loaded successfully on CUDA!
2026-07-27 22:35:33 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-07-27 22:35:38 - Ciri - INFO - [DeepSeek Cost] call=1, input_tokens=403, output_tokens=26, total_tokens=429, generation_time_seconds=5.7061
2026-07-27 22:35:38 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=1, input_tokens=403, output_tokens=26, total_tokens=429, generation_time_seconds=5.7061
2026-07-27 22:35:43 - Ciri - INFO - [DeepSeek Cost] call=2, input_tokens=403, output_tokens=26, total_tokens=429, generation_time_seconds=4.8471
2026-07-27 22:35:43 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=2, input_tokens=806, output_tokens=52, total_tokens=858, generation_time_

In [ ]:
!python -m ciri.ciri_eng \
  --input_path "$DATA_ROOT/$BENCHMARK/correct" \
  --output_path "$OUTPUT_ROOT/correct" \
  --model "$MODEL" \
  --system ext4 \
  --version 1.47.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml \
  --verbose

2026-07-27 22:38:08 - Ciri - INFO - Using device: CUDA
2026-07-27 22:38:08 - Ciri - INFO - Using dtype: torch.bfloat16
2026-07-27 22:38:08 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
Loading weights: 100% 291/291 [00:54<00:00,  5.34it/s]
2026-07-27 22:39:07 - Ciri - INFO - Model loaded successfully on CUDA!
2026-07-27 22:39:07 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-07-27 22:39:13 - Ciri - INFO - [DeepSeek Cost] call=1, input_tokens=390, output_tokens=26, total_tokens=416, generation_time_seconds=6.1713
2026-07-27 22:39:13 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=1, input_tokens=390, output_tokens=26, total_tokens=416, generation_time_seconds=6.1713
2026-07-27 22:39:18 - Ciri - INFO - [DeepSeek Cost] call=2, input_tokens=390, output_tokens=26, total_tokens=416, generation_time_seconds=5.4659
2026-07-27 22:39:18 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=2, input_tokens=780, output_tokens=52, total_tokens=832, generation_time_

In [ ]:
!python icse25_data/script/result_parser.py \
  --project "$BENCHMARK" \
  --model "$MODEL" \
  --mode "$MODE"

[Ciri Result] on ext4_CPD with deepseek-coder-6.7b-instruct and zero_shot mode
File-Level: Precision: N.A., Recall: 0.00, Accuracy: 0.50, F1: N.A.
Param-Level: Precision: N.A., Recall: 0.00, Accuracy: 0.93, F1: N.A.


CCD

In [25]:
BENCHMARK = "ext4_CCD"
MODEL = "deepseek-coder-6.7b-instruct"
MODE = "zero_shot"

OUTPUT_ROOT = f"icse25_data/results/synthesize_config/{BENCHMARK}/{MODEL}/{MODE}"

In [26]:
DRIVE_DATA = "/content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset/EXT4_XML_Dataset"

In [27]:
!cp -r "$DRIVE_DATA/ext4_CCD" \
    icse25_data/datasets/synthesize_config/

!cp "$DRIVE_DATA/Groundtruth/ext4_CCD.tsv" \
    icse25_data/datasets/synthesize_config/ground_truth/

In [28]:
!python -m ciri.ciri_eng \
  --input_path "icse25_data/datasets/synthesize_config/$BENCHMARK/erroneous" \
  --output_path "$OUTPUT_ROOT/erroneous" \
  --model "$MODEL" \
  --system ext4 \
  --version 1.47.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml \
  --verbose

2026-08-03 19:48:04 - Ciri - INFO - Using device: CUDA
2026-08-03 19:48:04 - Ciri - INFO - Using dtype: torch.bfloat16
2026-08-03 19:48:04 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
Loading weights: 100% 291/291 [00:53<00:00,  5.45it/s]
2026-08-03 19:49:02 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-03 19:49:02 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-08-03 19:49:08 - Ciri - INFO - [DeepSeek Cost] call=1, input_tokens=634, output_tokens=26, total_tokens=660, generation_time_seconds=6.4592
2026-08-03 19:49:08 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=1, input_tokens=634, output_tokens=26, total_tokens=660, generation_time_seconds=6.4592
2026-08-03 19:49:14 - Ciri - INFO - [DeepSeek Cost] call=2, input_tokens=634, output_tokens=26, total_tokens=660, generation_time_seconds=5.5989
2026-08-03 19:49:14 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=2, input_tokens=1268, output_tokens=52, total_tokens=1320, generation_tim

In [29]:
!python -m ciri.ciri_eng \
  --input_path "icse25_data/datasets/synthesize_config/$BENCHMARK/correct" \
  --output_path "$OUTPUT_ROOT/correct" \
  --model "$MODEL" \
  --system ext4 \
  --version 1.47.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml \
  --verbose

2026-08-03 20:08:55 - Ciri - INFO - Using device: CUDA
2026-08-03 20:08:55 - Ciri - INFO - Using dtype: torch.bfloat16
2026-08-03 20:08:55 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
Loading weights: 100% 291/291 [00:53<00:00,  5.40it/s]
2026-08-03 20:09:53 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-03 20:09:53 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-08-03 20:10:00 - Ciri - INFO - [DeepSeek Cost] call=1, input_tokens=613, output_tokens=26, total_tokens=639, generation_time_seconds=6.6527
2026-08-03 20:10:00 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=1, input_tokens=613, output_tokens=26, total_tokens=639, generation_time_seconds=6.6527
2026-08-03 20:10:05 - Ciri - INFO - [DeepSeek Cost] call=2, input_tokens=613, output_tokens=26, total_tokens=639, generation_time_seconds=5.7898
2026-08-03 20:10:05 - Ciri - INFO - [DeepSeek Cost Cumulative] calls=2, input_tokens=1226, output_tokens=52, total_tokens=1278, generation_tim

In [ ]:
!find icse25_data/datasets/synthesize_config/ext4_CCD \
  -type f | grep -E '/58(\.|$)'

In [ ]:
!ls icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct/zero_shot/correct/58

!ls icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct/zero_shot/erroneous/58

ls: cannot access 'icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct/zero_shot/correct/58': No such file or directory
ls: cannot access 'icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct/zero_shot/erroneous/58': No such file or directory


In [ ]:
!find "$OUTPUT_ROOT/correct" -maxdepth 1 -type f | wc -l
!find "$OUTPUT_ROOT/erroneous" -maxdepth 1 -type f | wc -l

57
57


In [ ]:
from pathlib import Path

input_root = Path(
    "icse25_data/datasets/synthesize_config/ext4_CCD"
)

output_root = Path(
    "icse25_data/results/synthesize_config/"
    "ext4_CCD/deepseek-coder-6.7b-instruct/zero_shot"
)

for category in ["correct", "erroneous"]:
    input_ids = {
        p.name for p in (input_root / category).iterdir()
        if p.is_file()
    }

    output_ids = {
        p.name for p in (output_root / category).iterdir()
        if p.is_file()
    } if (output_root / category).exists() else set()

    missing = sorted(
        input_ids - output_ids,
        key=lambda x: int(x) if x.isdigit() else x
    )

    print(f"\n{category.upper()}")
    print("Input files:", len(input_ids))
    print("Result files:", len(output_ids))
    print("Missing result IDs:", missing)


CORRECT
Input files: 57
Result files: 57
Missing result IDs: []

ERRONEOUS
Input files: 57
Result files: 57
Missing result IDs: []


In [30]:
!python icse25_data/script/result_parser.py \
  --project "$BENCHMARK" \
  --model "$MODEL" \
  --mode "$MODE"

[Ciri Result] on ext4_CCD with deepseek-coder-6.7b-instruct and zero_shot mode
File-Level: Precision: 1.00, Recall: 0.02, Accuracy: 0.51, F1: 0.03
Param-Level: Precision: 0.50, Recall: 0.02, Accuracy: 0.94, F1: 0.03


In [ ]:
!mkdir -p "/content/drive/MyDrive/CIRI_EXT4/DeepSeek_results"

!cp -r \
icse25_data/results/synthesize_config/ext4_SD/deepseek-coder-6.7b-instruct \
"/content/drive/MyDrive/CIRI_EXT4/DeepSeek_results/"

!cp -r \
icse25_data/results/synthesize_config/ext4_CPD/deepseek-coder-6.7b-instruct \
"/content/drive/MyDrive/CIRI_EXT4/DeepSeek_results/"

!cp -r \
icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct \
"/content/drive/MyDrive/CIRI_EXT4/DeepSeek_results/"

In [31]:
!mkdir -p "/content/drive/MyDrive/CIRI_EXT4/DeepSeek_results"

In [32]:
!cp -r \
icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct \
"/content/drive/MyDrive/CIRI_EXT4/DeepSeek_results/"